# Adaptive Trust Gate — any dataset, unattended

**Set the dataset in the next cell, then Runtime → Run all.** Approve the Drive popup
in section 2; nothing after that asks for anything.

Runs, in order: download → normalise → full single-seed pipeline (all 13 models:
experts, gates 3–8 and the four stacking baselines, plus comparison and interpretability)
→ multi-seed run for error bars.

Results copy themselves to Drive every 3 minutes, into
`MyDrive/atg_results/run_<dataset>_<time>/`. If the runtime dies, re-open this notebook,
keep the same dataset setting and Run all: completed seeds are restored **from runs of the
same dataset only** and skipped.

| `DATASET` | what it is | rough size |
|---|---|---|
| `movielens1m` | MovieLens-1M, dense, genre metadata | 1.0M ratings |
| `amazon` | Amazon Reviews 2023, one category (default CDs & Vinyl) | 4.8M ratings |
| `goodreads` | UCSD Book Graph, one genre (default poetry) | 1.2M ratings |


In [ ]:
# ---- choose the dataset -------------------------------------------------------
DATASET = 'movielens1m'          # 'movielens1m' | 'amazon' | 'goodreads'
AMAZON_CATEGORY = 'CDs_and_Vinyl'
GOODREADS_GENRE = 'poetry'
SEEDS = '42,1,2'
RUN_SINGLE_SEED = True           # False skips straight to the seed run
# --------------------------------------------------------------------------------
import os
os.environ.update(ATG_DATASET=DATASET, ATG_AMAZON_CATEGORY=AMAZON_CATEGORY,
                  ATG_GOODREADS_GENRE=GOODREADS_GENRE, ATG_SEEDS=SEEDS, PYTHONPATH='src')
print({k: os.environ[k] for k in ('ATG_DATASET', 'ATG_AMAZON_CATEGORY', 'ATG_GOODREADS_GENRE', 'ATG_SEEDS')})


## 1. Code and dependencies


In [ ]:
%cd /content
!git clone -q https://github.com/Ar555Rathod/adaptive-trust-gate.git 2>/dev/null || git -C adaptive-trust-gate pull -q origin master
%cd /content/adaptive-trust-gate
!git log --oneline -1
!pip install -q -r requirements.txt
import surprise; print('scikit-surprise OK:', surprise.__version__)


In [ ]:
import sys; sys.path.insert(0, 'src')
from atg import config
SLUG = config.DATASET_SLUG
print('dataset slug:', SLUG, '| results ->', config.RESULTS_DIR)


## 2. Mount Drive — the only prompt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Auto-save to Drive every 3 minutes (additive; existing folders untouched)


In [ ]:
import shutil, subprocess, threading, time
from pathlib import Path

SRC = Path('/content/adaptive-trust-gate/results')
RUN_ID = time.strftime(f'run_{SLUG}_%Y%m%d_%H%M')
DST = Path('/content/drive/MyDrive/atg_results') / RUN_ID
DST.mkdir(parents=True, exist_ok=True)
HAVE_RSYNC = shutil.which('rsync') is not None
sync_state = {'count': 0, 'last': None, 'error': None}

def sync_once():
    if not SRC.exists() or not any(SRC.rglob('*')):
        return False
    if HAVE_RSYNC:
        subprocess.run(['rsync', '-a', f'{SRC}/', f'{DST}/'], check=False, capture_output=True)
    else:
        shutil.copytree(SRC, DST, dirs_exist_ok=True)
    return True

def sync_loop(every=180):
    while True:
        try:
            if sync_once():
                sync_state['count'] += 1
                sync_state['last'] = time.strftime('%H:%M:%S')
        except Exception as e:
            sync_state['error'] = repr(e)
        time.sleep(every)

threading.Thread(target=sync_loop, daemon=True).start()
print(f'auto-save ON  ->  {DST}')


## 4. Download and normalise

Files come straight from UCSD / GroupLens — nothing needs uploading.


In [ ]:
!python scripts/00_download_data.py
!python src/atg/data/normalize.py


## 5. Single-seed pipeline (seed 42)

Skipped when `RUN_SINGLE_SEED = False`.


In [ ]:
def run(*scripts):
    if not RUN_SINGLE_SEED:
        print('skipped (RUN_SINGLE_SEED = False)'); return
    for s in scripts:
        print(f'\n===== {s} =====', flush=True)
        r = subprocess.run(['python', '-u', '-W', 'ignore', s])
        if r.returncode != 0:
            print(f'!! {s} exited {r.returncode} -- continuing so the seed run still happens')

run('scripts/01_build_splits.py', 'scripts/02_train_experts.py')


In [ ]:
run('scripts/03_static_hybrid.py', 'scripts/04_learned_gate.py', 'scripts/05_bandit_gate.py')


In [ ]:
run('scripts/06_ga_gate.py', 'scripts/07_sequential_gate.py')


In [ ]:
run('scripts/13_calibrated_gate.py', 'scripts/11_external_baselines.py', 'scripts/08_full_comparison.py', 'scripts/09_interpretability.py')


## 6. Multi-seed run

All 13 models refitted per seed; checkpoints after each seed. Only checkpoints from earlier
runs **of this same dataset** are restored, and a checkpoint missing any current model is set
aside by the script rather than resumed.


In [ ]:
import json as _json
root = Path('/content/drive/MyDrive/atg_results')
dest = config.METRICS_DIR
dest.mkdir(parents=True, exist_ok=True)
prev = sorted((q for q in root.glob(f'run_*/{SLUG}/metrics/multiseed_full_comparison.json')
               if q.parent.parent.parent.name != RUN_ID),
              key=lambda q: q.stat().st_mtime, reverse=True)
if prev:
    shutil.copy2(prev[0], dest / 'multiseed_full_comparison.json')
    print(f'restored checkpoint from {prev[0].parent.parent.parent.name}:',
          'seeds', _json.load(open(prev[0])).get('seeds', []))
else:
    print(f'no earlier {SLUG} checkpoint in Drive -- starting the seed run from scratch')


In [ ]:
for attempt in range(1, 6):
    print(f'--- attempt {attempt}  ({time.strftime("%H:%M:%S")}) ---', flush=True)
    r = subprocess.run(['python', '-u', '-W', 'ignore', 'scripts/10_multiseed_full.py'])
    if r.returncode == 0:
        print('multiseed finished cleanly'); break
    print(f'exit {r.returncode} -- retrying from the last checkpoint')
    sync_once()
else:
    print('did not finish after 5 attempts -- completed seeds are checkpointed and saved')


In [ ]:
!python scripts/12_seed_paired_analysis.py


## 7. Save and check


In [ ]:
print('final sync:', 'done' if sync_once() else 'nothing to copy')
print(f'syncs: {sync_state["count"]}   last: {sync_state["last"]}   error: {sync_state["error"]}')
print(f'saved to: {DST}')
p = dest / 'multiseed_full_comparison.json'
if p.exists():
    d = _json.load(open(p))
    print('\nseeds completed:', d['seeds'])
    for name, agg in sorted(d['aggregate'].items(), key=lambda kv: kv[1]['overall']['rmse_mean']):
        a = agg['overall']; print(f"  {name:22s} {a['rmse_mean']:.4f} +/- {a['rmse_std']:.4f}")
